# Gnod Song Recommender — Data Pipeline & Model Evaluation

This notebook consolidates the end-to-end data pipeline and unsupervised machine learning workflow for the **Gnod Song Recommender** project.

### Project Pipeline
1. **Setup & Configuration** — Imports, API authentication, and project constants.
2. **Web Scraping (Billboard Hot 100)** — Scraping current chart hits into `billboard_hot100.csv` for taste-independent popularity recommendations.
3. **API Data Collection (Spotify + ReccoBeats)** — Building and cleaning the multi-genre audio-feature catalog (`gnod_songs.csv`) with rate-limit protection and CSV caching.
4. **Feature Scaling & Clustering Evaluation** — Comparing K-Means, Agglomerative Clustering, and DBSCAN on 9 continuous audio features using internal metrics, cluster balance, and Adjusted Rand Index (ARI).
5. **Cluster Interpretation & Recommender Testing** — Inspecting cluster profiles, borderline genres, and benchmarking same-genre hit rates across four recommendation strategies.

## 1. Setup & Imports

Ensure you have a `.env` file in the project root containing your Spotify Developer credentials (and that `.env` is listed in `.gitignore`):

```env
SPOTIPY_CLIENT_ID=your_client_id
SPOTIPY_CLIENT_SECRET=your_client_secret
```

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import IFrame, display

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    pairwise_distances,
)

# Load environment variables from .env
load_dotenv()

# Initialize Spotipy client (status_retries=0 fails fast on rate limits instead of hanging)
sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=os.getenv("SPOTIPY_CLIENT_ID"),
        client_secret=os.getenv("SPOTIPY_CLIENT_SECRET"),
    ),
    status_retries=0,
)

# Project file paths and API rate-limit settings
BILLBOARD_FILE = "billboard_hot100.csv"
GNOD_FILE = "gnod_songs.csv"
HOT100_FEATURES_FILE = "hot100_songs.csv"

SEARCH_WAIT = 1    # Seconds between Spotify API requests
FEATURES_WAIT = 1  # Seconds between ReccoBeats API requests

# 16 distinct genres chosen for strong sonic contrast and reliable ReccoBeats coverage
GENRES = [
    "rock", "jazz", "hip-hop", "electronic",
    "reggae", "country", "blues", "metal",
    "pop", "latin", "soul", "funk",
    "techno", "punk", "folk", "flamenco",
]
SONGS_PER_GENRE = 100

# 9 continuous audio features (key and mode excluded to avoid splitting similar songs by major/minor key)
FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]

## 2. Web Scraping: Billboard Hot 100

Scrape the current Billboard Hot 100 chart (rank, title, and artist) using `requests` and `BeautifulSoup`. To avoid unnecessary repeat requests, the chart is loaded from `billboard_hot100.csv` if it already exists locally.

In [ ]:
def scrape_billboard_hot100(url="https://www.billboard.com/charts/hot-100/"):
    """Scrape the current Billboard Hot 100 chart into a DataFrame."""
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")
    rows = soup.select("ul.o-chart-results-list-row")

    records = []
    for idx, row in enumerate(rows, start=1):
        title_tag = row.select_one("h3#title-of-a-story")
        if not title_tag:
            continue
        artist_tag = title_tag.find_next_sibling("span")
        records.append({
            "rank": idx,
            "title": title_tag.get_text(strip=True),
            "artist": artist_tag.get_text(strip=True) if artist_tag else "",
        })

    return pd.DataFrame(records)


# Load cached chart or scrape once
if os.path.exists(BILLBOARD_FILE):
    billboard_df = pd.read_csv(BILLBOARD_FILE)
    print(f"Loaded {len(billboard_df)} tracks from {BILLBOARD_FILE}")
else:
    billboard_df = scrape_billboard_hot100()
    billboard_df.to_csv(BILLBOARD_FILE, index=False)
    print(f"Scraped and saved {len(billboard_df)} tracks to {BILLBOARD_FILE}")

billboard_df.head()

In [ ]:
def check_hot100_song(song_title, chart_df=billboard_df):
    """Check if a song is in the Billboard Hot 100 and recommend another chart track."""
    song_clean = song_title.strip().lower()
    titles_clean = chart_df["title"].str.strip().str.lower()

    if song_clean in titles_clean.values:
        other_songs = chart_df[titles_clean != song_clean]
        pick = other_songs.sample(1).iloc[0]
        print(f"Nice pick! '{song_title.strip()}' is trending.")
        print(f"You might also like: {pick['title']} by {pick['artist']}")
    else:
        print("That song is not currently in the Billboard Hot 100 — switching to acoustic similarity!")


# Quick sanity check
if not billboard_df.empty:
    check_hot100_song(billboard_df["title"].iloc[0])

## 3. API Data Collection: Spotify + ReccoBeats

Because Spotify deprecated `audio_features` and restricted playlist endpoints for new developer apps, we:
1. Collect tracks across **16 diverse genres** using paginated `sp.search(q="genre:...")` calls (`limit=10`).
2. Fetch continuous audio features in batches of up to **40 track IDs** from the free [ReccoBeats API](https://api.reccobeats.com/v1/audio-features).
3. Save progress after every genre to `gnod_songs.csv` so collection can resume safely without exceeding Spotify's request quota.

In [ ]:
def play_song(track_id):
    """Render an embedded Spotify player for a track ID inside Jupyter."""
    return IFrame(
        src=f"https://open.spotify.com/embed/track/{track_id}",
        width="320",
        height="80",
        frameborder="0",
        allowtransparency="true",
        allow="encrypted-media",
    )


def audio_features(tracks):
    """Fetch audio features from ReccoBeats for up to 40 Spotify track IDs or URIs.
    Tracks not found in ReccoBeats are skipped; results are matched by Spotify ID.
    """
    if isinstance(tracks, str):
        tracks = [tracks]
    if not tracks:
        return []
    if len(tracks) > 40:
        raise ValueError("ReccoBeats accepts at most 40 tracks per request.")

    # Strip URI prefixes (e.g. 'spotify:track:<id>') if present
    ids = [t.split(":")[-1] for t in tracks]
    response = requests.get(
        "https://api.reccobeats.com/v1/audio-features",
        params={"ids": ",".join(ids)},
        timeout=15,
    )
    response.raise_for_status()

    # Map each returned entry back to its Spotify track ID via the 'href' URL
    found = {
        feat["href"].split("/")[-1]: feat
        for feat in response.json().get("content", [])
    }
    return [{**found[i], "id": i} for i in ids if i in found]

In [ ]:
def collect_genre(genre, n_songs=SONGS_PER_GENRE):
    """Collect up to n_songs for a single genre from Spotify and merge with ReccoBeats features."""
    tracks = []
    while len(tracks) < n_songs:
        results = sp.search(
            q=f"genre:{genre}",
            type="track",
            limit=10,
            offset=len(tracks),
            market="DE",
        )["tracks"]
        if not results["items"]:
            break
        tracks.extend(results["items"])
        time.sleep(SEARCH_WAIT)

    tracks_df = pd.DataFrame({
        "id": [track["id"] for track in tracks],
        "song": [track["name"] for track in tracks],
        "artist": [track["artists"][0]["name"] for track in tracks],
        "genre": genre,
    })

    # Request ReccoBeats audio features in chunks of 40 IDs
    features = []
    track_ids = tracks_df["id"].tolist()
    for i in range(0, len(track_ids), 40):
        chunk = track_ids[i : i + 40]
        features.extend(audio_features(chunk))
        time.sleep(FEATURES_WAIT)

    features_df = pd.DataFrame(features)
    if features_df.empty:
        return pd.DataFrame()

    return tracks_df.merge(features_df, on="id", how="inner")


def build_dataset(genres, n_songs, file_name):
    """Collect all genres incrementally, skipping genres already saved in file_name."""
    dataset = (
        pd.read_csv(file_name)
        if os.path.exists(file_name)
        else pd.DataFrame(columns=["id", "genre"])
    )

    for genre in genres:
        if genre in dataset["genre"].values:
            print(f"{genre}: already collected, skipping")
            continue
        try:
            genre_df = collect_genre(genre, n_songs)
        except (spotipy.SpotifyException, requests.HTTPError) as e:
            print(f"Stopped at {genre} ({e}). Progress saved — retry later.")
            break
        if genre_df.empty:
            print(f"{genre}: no tracks with audio features found")
            continue

        dataset = pd.concat([dataset, genre_df], ignore_index=True).drop_duplicates(subset="id")
        dataset.to_csv(file_name, index=False)
        print(f"{genre}: +{len(genre_df)} songs ({len(dataset)} total)")

    return dataset

In [ ]:
# Build or load the Gnod catalog
gnod_df = build_dataset(GENRES, SONGS_PER_GENRE, GNOD_FILE)

# Remove noisy/low-coverage genres if present from earlier collection runs
excluded_genres = ["classical", "acoustic", "k-pop"]
if gnod_df["genre"].isin(excluded_genres).any():
    gnod_df = gnod_df[~gnod_df["genre"].isin(excluded_genres)].reset_index(drop=True)
    gnod_df.to_csv(GNOD_FILE, index=False)

print(f"Final catalog shape: {gnod_df.shape}")
gnod_df["genre"].value_counts()

### 3.1 Optional: Enriching Billboard Hot 100 with Audio Features

Match scraped Billboard Hot 100 tracks to Spotify IDs and fetch their ReccoBeats audio features (cached in `hot100_songs.csv`). Note that brand-new releases may not yet appear in ReccoBeats.

In [ ]:
def find_spotify_id(title, artist):
    """Return the Spotify track ID of the top search match, or None if not found."""
    res = sp.search(q=f"{title} {artist}", type="track", limit=1, market="DE")
    items = res["tracks"]["items"]
    return items[0]["id"] if items else None


if os.path.exists(HOT100_FEATURES_FILE):
    hot100_df = pd.read_csv(HOT100_FEATURES_FILE)
    print(f"Loaded {len(hot100_df)} enriched Hot 100 tracks from {HOT100_FEATURES_FILE}")
elif not billboard_df.empty:
    ids = []
    for title, artist in zip(billboard_df["title"], billboard_df["artist"]):
        ids.append(find_spotify_id(title, artist))
        time.sleep(SEARCH_WAIT)

    hot100_meta = pd.DataFrame({
        "id": ids,
        "song": billboard_df["title"],
        "artist": billboard_df["artist"],
        "genre": "hot100",
    }).dropna(subset=["id"])

    hot100_feats = []
    hot100_ids = hot100_meta["id"].tolist()
    for i in range(0, len(hot100_ids), 40):
        hot100_feats.extend(audio_features(hot100_ids[i : i + 40]))
        time.sleep(FEATURES_WAIT)

    hot100_df = hot100_meta.merge(pd.DataFrame(hot100_feats), on="id", how="inner")
    hot100_df.to_csv(HOT100_FEATURES_FILE, index=False)
    print(f"Saved {len(hot100_df)} enriched Hot 100 tracks to {HOT100_FEATURES_FILE}")

## 4. Clustering & Model Evaluation on `gnod_songs.csv`

**Key Modeling Rules:**
- **Unsupervised training:** The `genre` column is **never** passed to the clustering models — it is held back as an external answer key to evaluate whether audio clusters make musical sense.
- **Feature selection:** We use the **9 continuous audio features** (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`) and standardize them with `StandardScaler` so high-magnitude columns like `tempo` and `loudness` do not dominate `0–1` features.

In [ ]:
# Load catalog and prepare standardized feature matrix X
songs = pd.read_csv(GNOD_FILE)
songs = songs.dropna(subset=FEATURES).drop_duplicates(subset="id").reset_index(drop=True)

scaler = StandardScaler()
X = scaler.fit_transform(songs[FEATURES])

print(f"Tracks ready for clustering: {len(songs)} | Features: {len(FEATURES)}")
songs[["song", "artist", "genre"] + FEATURES].head()

In [ ]:
def cluster_metrics(X_data, labels):
    """Compute internal clustering metrics (Silhouette, Calinski-Harabasz, Davies-Bouldin)
    while excluding DBSCAN noise points (-1).
    """
    kept = labels != -1
    X_clean, labels_clean = X_data[kept], labels[kept]
    n_clusters = len(np.unique(labels_clean))

    if n_clusters < 2:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
        }

    return {
        "silhouette": silhouette_score(X_clean, labels_clean),
        "calinski_harabasz": calinski_harabasz_score(X_clean, labels_clean),
        "davies_bouldin": davies_bouldin_score(X_clean, labels_clean),
    }


def plot_silhouette(X_data, labels, title="", ax=None):
    """Plot point-level silhouette bars grouped by cluster with a dashed mean line."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))

    values = silhouette_samples(X_data, labels)
    y = 0
    for cluster in np.unique(labels):
        cluster_values = np.sort(values[labels == cluster])
        ax.fill_betweenx(
            np.arange(y, y + len(cluster_values)),
            0,
            cluster_values,
            color=plt.cm.tab10(cluster % 10),
            alpha=0.75,
        )
        ax.text(-0.04, y + len(cluster_values) / 2, str(cluster), ha="right", va="center")
        y += len(cluster_values) + 12

    ax.axvline(values.mean(), color="black", linestyle="--", label=f"Mean = {values.mean():.3f}")
    ax.set(
        title=title,
        xlabel="Silhouette value",
        yticks=[],
        xlim=(min(-0.2, values.min() - 0.05), 1),
    )
    ax.legend(loc="lower right")
    return ax

### 4.1 Choosing `k` for K-Means

We sweep `k = 2` through `15` across all three internal metrics:
- **Silhouette** (higher is better)
- **Calinski-Harabasz** (higher is better)
- **Davies-Bouldin** (lower is better)

While `k = 3` peaks on raw Silhouette as very broad mood buckets, **`k = 6`** hits the **global minimum on Davies-Bouldin (~1.47)** with a local **Silhouette rebound (~0.178)** and strong Calinski-Harabasz (~186), offering the best practical balance for distinct musical vibes.

In [ ]:
ks = range(2, 16)
kmeans_scores = pd.DataFrame(
    [
        cluster_metrics(X, KMeans(n_clusters=k, random_state=42, n_init="auto").fit_predict(X))
        for k in ks
    ],
    index=ks,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
metric_titles = {
    "silhouette": "Silhouette (higher is better)",
    "calinski_harabasz": "Calinski-Harabasz (higher is better)",
    "davies_bouldin": "Davies-Bouldin (lower is better)",
}

for ax, (metric, title) in zip(axes, metric_titles.items()):
    ax.plot(ks, kmeans_scores[metric], marker="o")
    ax.axvline(6, color="crimson", linestyle="--", alpha=0.6, label="Selected k=6")
    ax.set(title=title, xlabel="Number of clusters (k)")
    ax.legend()

plt.tight_layout()
plt.show()

### 4.2 Comparing Clustering Models (Internal Metrics, Balance & ARI)

We compare **K-Means (`k=6`)**, **Agglomerative Clustering** (`ward`, `complete`, `average` at `k=6`), and **DBSCAN** (`eps=1.5` and `eps=1.8`). Alongside internal scores, we track:
- **`largest_%`** — Share of non-noise tracks in the largest cluster (catches single-mega-cluster collapse).
- **`noise_%`** — Share of tracks dropped as `-1` noise (unusable for full-catalog recommendations).
- **`ARI` (Adjusted Rand Index)** — External alignment against the hidden `genre` labels.

In [ ]:
BEST_K = 6

models = {
    "K-Means (k=6)": KMeans(n_clusters=BEST_K, random_state=42, n_init="auto"),
    "Agglomerative (ward)": AgglomerativeClustering(n_clusters=BEST_K, linkage="ward"),
    "Agglomerative (complete)": AgglomerativeClustering(n_clusters=BEST_K, linkage="complete"),
    "Agglomerative (average)": AgglomerativeClustering(n_clusters=BEST_K, linkage="average"),
    "DBSCAN (eps=1.5)": DBSCAN(eps=1.5, min_samples=5),
    "DBSCAN (eps=1.8)": DBSCAN(eps=1.8, min_samples=5),
}

rows = []
all_labels = {}

for name, model in models.items():
    labels = model.fit_predict(X)
    all_labels[name] = labels

    non_noise = labels[labels != -1]
    noise_pct = (labels == -1).mean() * 100
    n_clusters = len(np.unique(non_noise))
    largest_pct = (
        pd.Series(non_noise).value_counts(normalize=True).max() * 100
        if len(non_noise) > 0
        else 0.0
    )

    metrics = cluster_metrics(X, labels)
    ari = adjusted_rand_score(songs["genre"], labels)

    rows.append({
        "model": name,
        "n_clusters": n_clusters,
        "noise_%": round(noise_pct, 1),
        "largest_%": round(largest_pct, 1),
        "silhouette": round(metrics["silhouette"], 4),
        "calinski_harabasz": round(metrics["calinski_harabasz"], 2),
        "davies_bouldin": round(metrics["davies_bouldin"], 4),
        "ARI": round(ari, 4),
    })

comparison = pd.DataFrame(rows).sort_values("ARI", ascending=False).reset_index(drop=True)
display(comparison)

**Model Selection Takeaway:**
- **Internal metrics vs. ARI do not fully agree:** Raw Silhouette and Davies-Bouldin reward degenerate models (`Agglomerative average/complete` and `DBSCAN`) that dump **86%–99%** of songs into one mega-cluster or drop up to **30.5%** of songs as noise, resulting in near-zero ARI.
- **Winner — `K-Means (k=6)`:** Keeps **0.0% noise**, maintains balanced cluster sizes (**~29.6% largest cluster**), leads on **Calinski-Harabasz** and **ARI**, and supports `.predict()` for classifying unseen songs in production.

## 5. Interpreting the Winning Model (`K-Means, k=6`)

We attach the winning K-Means cluster labels to `songs`, inspect point-level silhouette values to spot borderline/crossover tracks, and reveal how the 16 genres map across the 6 sonic clusters.

In [ ]:
# Lock in winning K-Means model
kmeans_labels = all_labels["K-Means (k=6)"]
songs["cluster"] = kmeans_labels
songs["silhouette"] = silhouette_samples(X, kmeans_labels)

# 1. Silhouette plot for K-Means (k=6)
fig, ax = plt.subplots(figsize=(7, 5))
plot_silhouette(X, kmeans_labels, title=f"K-Means (k={BEST_K}) — Point-Level Silhouette", ax=ax)
plt.tight_layout()
plt.show()

# 2. Cluster sizes and mean audio feature profiles
cluster_profile = songs.groupby("cluster")[FEATURES].mean().round(3)
cluster_profile.insert(0, "n_songs", songs["cluster"].value_counts().sort_index())
print("--- Cluster Profiles (Mean Audio Features) ---")
display(cluster_profile)

In [ ]:
# 3. The Genre Reveal: cross-tabulate hidden genre labels against K-Means clusters
print("--- Genre Distribution Across Clusters ---")
display(pd.crosstab(songs["genre"], songs["cluster"]))

# 4. Hardest songs & genres to place (lowest silhouette values)
print("--- Lowest Silhouette Songs (Borderline / Crossover Tracks) ---")
display(songs.nsmallest(10, "silhouette")[["song", "artist", "genre", "cluster", "silhouette"]])

print("--- Average Silhouette by Genre (Hardest Genres First) ---")
display(songs.groupby("genre")["silhouette"].mean().sort_values().round(3))

## 6. Testing the Gnod Song Recommender

We evaluate four recommendation strategies on the full `gnod_songs.csv` catalog using **same-genre hit rate** (fraction of the `n=5` recommendations that share the seed track's hidden genre):
1. **`closest_cluster`**: `n` nearest Euclidean neighbors inside the seed song's K-Means cluster.
2. **`closest_all`**: `n` nearest Euclidean neighbors across the entire catalog.
3. **`random_cluster`**: `n` random songs from the same K-Means cluster (more discovery/variety).
4. **`random_all`**: `n` random songs from the entire catalog (baseline).

In [ ]:
# Precompute pairwise Euclidean distances in standardized audio-feature space
DIST_MATRIX = pairwise_distances(X, metric="euclidean")
RNG = np.random.default_rng(42)


def recommend_ids(i, n=5, method="closest_cluster"):
    """Return positional indices of n recommended tracks for song index i."""
    same_cluster = songs["cluster"].values == songs.loc[i, "cluster"]
    not_self = np.arange(len(songs)) != i

    if method == "closest_cluster":
        candidates = np.where(same_cluster & not_self)[0]
        order = np.argsort(DIST_MATRIX[i, candidates])
        return candidates[order[:n]]

    if method == "closest_all":
        candidates = np.where(not_self)[0]
        order = np.argsort(DIST_MATRIX[i, candidates])
        return candidates[order[:n]]

    if method == "random_cluster":
        candidates = np.where(same_cluster & not_self)[0]
        return RNG.choice(candidates, size=min(n, len(candidates)), replace=False)

    if method == "random_all":
        candidates = np.where(not_self)[0]
        return RNG.choice(candidates, size=min(n, len(candidates)), replace=False)

    raise ValueError(f"Unknown method: {method}")


def recommend(song_name, n=5, method="closest_cluster"):
    """Find a song by title (case-insensitive) and return n recommendations."""
    matches = songs.index[songs["song"].str.strip().str.lower() == song_name.strip().lower()]
    if len(matches) == 0:
        print(f"Song '{song_name}' not found in catalog.")
        return pd.DataFrame(columns=["song", "artist", "genre", "cluster"])

    idx = matches[0]
    rec_indices = recommend_ids(idx, n=n, method=method)
    return songs.loc[rec_indices, ["song", "artist", "genre", "cluster"]]

In [ ]:
# Benchmark same-genre hit rate across all songs in the catalog (n=5)
def evaluate_hit_rate(method, n=5):
    """Compute mean same-genre hit rate across every track in the catalog."""
    hits = []
    genres = songs["genre"].values
    for i in range(len(songs)):
        recs = recommend_ids(i, n=n, method=method)
        hits.append((genres[recs] == genres[i]).mean())
    return float(np.mean(hits))


methods = ["closest_all", "closest_cluster", "random_cluster", "random_all"]
hit_rates = pd.DataFrame({
    "method": methods,
    "same_genre_hit_rate": [round(evaluate_hit_rate(m, n=5), 4) for m in methods],
})
hit_rates["lift_vs_random"] = (
    hit_rates["same_genre_hit_rate"]
    / hit_rates.loc[hit_rates["method"] == "random_all", "same_genre_hit_rate"].values[0]
).round(2)

display(hit_rates)

# Bar chart of hit rates
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(hit_rates["method"], hit_rates["same_genre_hit_rate"] * 100, color=["#2563EB", "#3B82F6", "#60A5FA", "#94A3B8"])
ax.set(title="Same-Genre Hit Rate by Recommendation Method (n=5)", ylabel="Hit Rate (%)", xlabel="Method")
for idx, row in hit_rates.iterrows():
    ax.text(idx, row["same_genre_hit_rate"] * 100 + 0.6, f"{row['same_genre_hit_rate']*100:.1f}%", ha="center")
plt.tight_layout()
plt.show()

In [ ]:
# Qualitative demo: compare 'random_cluster' vs 'closest_cluster' on a sample track
sample_song = songs["song"].iloc[0]
print(f"Seed track: {sample_song} — {songs['artist'].iloc[0]} ({songs['genre'].iloc[0]})\n")

print("--- Closest Cluster Recommendations ---")
display(recommend(sample_song, n=5, method="closest_cluster"))

print("--- Random Cluster Recommendations (More Variety) ---")
display(recommend(sample_song, n=5, method="random_cluster"))

# Preview the seed track using the embedded Spotify player
play_song(songs["id"].iloc[0])